# Workshop — Final Project: Orders Medallion Pipeline

Build a Bronze → Silver → Gold pipeline for raw JSON orders and run it as a scheduled Lakeflow Job.

**Learning objectives**
- Implement a 3-layer Medallion pipeline: Bronze → Silver → Gold
- Ingest raw JSON into a Bronze Delta table with metadata columns, then clean and enrich it in Silver (cast, enrich, filter)
- Aggregate daily revenue KPIs into a Gold Delta table
- Wire the three notebooks into a Lakeflow Job with task dependencies and job parameters
- Add a scheduled trigger and run the pipeline end to end on job compute (Serverless)

**Prerequisites:** M03–M06 and the Ingestion workshop. Section 2 runs `%run ../../setup/00_setup`.

---

## 1. Scenario

You are a data engineer at a retail company. Your manager needs a **fully automated pipeline** that turns raw JSON order files into a daily revenue report, ready for the analytics team by 06:00 every morning.

You implement the pipeline as three notebooks that follow the **Medallion architecture**, then wire them together into a **Lakeflow Job** (formerly Workflows).

![Orders Medallion pipeline: source, Bronze, Silver and Gold](../../../assets/images/orders_medallion_pipeline.png "Orders Medallion pipeline")

### Layer responsibilities

| Layer | Table | Rule | Expected rows |
|-------|-------|------|---------------|
| **Bronze** | `bronze.lab_orders` | Raw data — only metadata columns added | 100,000 |
| **Silver** | `silver.lab_orders` | Typed, enriched, filtered | 90,904 |
| **Gold** | `gold.lab_daily_orders` | Aggregated — one row per day | 32 |

### Workshop files

| Notebook | Layer | What you build |
|----------|-------|----------------|
| `tasks/lab_task_01_bronze` | Bronze | Raw JSON → Delta *(pre-filled — read and run)* |
| `tasks/lab_task_02_silver` | Silver | Casting, enrichment, filtering *(TODO + CHECK)* |
| `tasks/lab_task_03_gold`   | Gold   | Daily revenue aggregation *(TODO + CHECK)* |

---

## 2. Verify setup

Run the two cells below. Keep the printed `catalog` and `source_path` values: you paste them into the job in step 5.3.

In [0]:
%run ../../setup/00_setup

In [0]:
# Job parameters for step 5.3 (Key = Value)
print(f"catalog     = {CATALOG}")
print(f"source_path = {DATASET_PATH}")

---

## 3. Complete the task notebooks

Open the `tasks` folder and work through the notebooks in order:

1. `lab_task_01_bronze` is **pre-filled** — read it and run it.
2. In `lab_task_02_silver` and `lab_task_03_gold`, replace each `# YOUR CODE HERE` with your code, then run the notebook top to bottom.

### How a task notebook works

- It starts with `%run ../../../setup/00_setup`.
- It reads its settings from `dbutils.widgets` (input fields at the top of the notebook): run it as-is.
- **CHECK** cells follow the key exercises: an `AssertionError` tells you what to fix.
- It ends with `assert` checks and `dbutils.notebook.exit(json)`; a failed assert fails the job task.

### If you fall behind

> **Tip:** If Silver or Gold still fails when the trainer calls time, switch to the solutions, so you still build and run the job.
> The trainer shares a read-only solutions folder in the workspace and tells you its path.
> In section 5, point the `silver_transform` and `gold_aggregation` tasks at `lab_task_02_silver_solution` and `lab_task_03_gold_solution` from that folder.
> Keep `bronze_load` on your own `lab_task_01_bronze`. Skip section 4: the job run in step 5.8 creates the Silver and Gold tables.

---

## 4. Smoke test

Run the cell below after you complete all three task notebooks. Expected row counts: Bronze 100,000 · Silver 90,904 · Gold 32.

In [0]:
# Rows per layer next to the expected count
display(spark.sql(f"""
    SELECT 'bronze' AS layer, COUNT(*) AS row_count, 100000 AS expected FROM {CATALOG}.{BRONZE_SCHEMA}.lab_orders
    UNION ALL SELECT 'silver', COUNT(*), 90904 FROM {CATALOG}.{SILVER_SCHEMA}.lab_orders
    UNION ALL SELECT 'gold', COUNT(*), 32 FROM {CATALOG}.{GOLD_SCHEMA}.lab_daily_orders
"""))

---

## 5. Create the Lakeflow Job

A Lakeflow Job runs notebooks as tasks, in a set order, on a schedule or on demand.

### 5.1 Create a new job

In the left sidebar click **Jobs & Pipelines**, then **Create new → Job**.

![Create Job](../../../assets/images/6f524625f8464c29a2572b0a324333a5.webp)

### 5.2 Set the job name

Enter a name you will recognise, for example `<your_name>_orders_pipeline`.

![Job Name Configuration](../../../assets/images/44ae3f0f9ebf4a8f838412613db941bd.png "Job Name Configuration")

> **Note:** The name does not need to be unique: Databricks identifies each job by its **Job ID**.

![Job Details — ID & Creator](../../../assets/images/460925af2b214f93a2252aed0ea86936.webp)

### 5.3 Configure job parameters

In the **Job details** panel on the right, click **Edit parameters** and add two parameters.
Copy the values from the section 2 output — do not type the catalog name by hand.

| Key | Value |
|-----|-------|
| `catalog` | the `catalog` value printed in section 2 |
| `source_path` | the `source_path` value printed in section 2 |

![Job Parameters](../../../assets/images/e24f8fbe50f642e0a28054972140dcb1.webp)

> **Note:** The screenshot shows example values; yours come from section 2.

**How job parameters reach the notebooks**

Each task notebook creates widgets whose defaults come from `00_setup`, for example `dbutils.widgets.text("catalog", CATALOG, "Catalog")`.
The job passes its parameters to every notebook task, and a job parameter with the same name **overrides** the widget default.
Widgets without a job parameter (`schema`, `bronze_schema`, `silver_schema`, `gold_schema`) keep their defaults.

### 5.4 Add the first task: `bronze_load`

Click **Add task** (or fill in the task form if the new job already shows one) and configure:

| Field | Value |
|-------|-------|
| **Task name** | `bronze_load` |
| **Type** | Notebook |
| **Source** | Workspace |
| **Path** | pick `lab_task_01_bronze` in the file browser (in your Git folder: `…/notebooks/workshops/final_labs/tasks/`) |
| **Compute** | Serverless |

![Add Task](../../../assets/images/8d2245e891b849e5966aae38a0e33c5f.webp)

![Add Task — bronze_load](../../../assets/images/a7d00efeb1bb409aaae40260a6485eff.png "Add Task — bronze_load")

> **Note:** Interactive notebooks run on all-purpose compute; scheduled jobs belong on job compute or Serverless.
> Job compute is billed at a lower DBU (Databricks Unit, the billing unit) rate than all-purpose compute, and Serverless has no idle cluster to pay for.
> If Serverless is not offered for jobs in your workspace, pick the compute your trainer names. On Databricks Free Edition only Serverless is available: use it for all three tasks.

Click **Create task** when done.

### 5.5 Add the second task: `silver_transform`

Click **Add task** and configure:

| Field | Value |
|-------|-------|
| **Task name** | `silver_transform` |
| **Type** | Notebook |
| **Source** | Workspace |
| **Path** | `lab_task_02_silver` (or `lab_task_02_silver_solution` from the trainer's solutions folder) |
| **Depends on** | `bronze_load` |
| **Compute** | Serverless |

> **Note:** **Depends on** makes `silver_transform` start only after `bronze_load` succeeds.

### 5.6 Add the third task: `gold_aggregation`

Click **Add task** and configure:

| Field | Value |
|-------|-------|
| **Task name** | `gold_aggregation` |
| **Type** | Notebook |
| **Source** | Workspace |
| **Path** | `lab_task_03_gold` (or `lab_task_03_gold_solution` from the trainer's solutions folder) |
| **Depends on** | `silver_transform` |
| **Compute** | Serverless |

Your completed pipeline DAG (directed acyclic graph: the tasks and the dependencies between them) should look like this:

![Pipeline DAG — three tasks in sequence](../../../assets/images/eecc585d0d5c400280ff3b31fdf05a3e.png "Pipeline DAG — three tasks in sequence")

### 5.7 Add a scheduled trigger

1. In the **Job details** panel on the right, under **Schedules & Triggers**, click **Add trigger**.
2. **Trigger type:** Scheduled.
3. **Schedule type:** Advanced — **Every Day at 06:00**, time zone **Europe/Warsaw**.
4. Optional: tick **Show cron syntax** to see the same schedule as a Quartz cron expression, `0 0 6 * * ?`
   (fields: seconds, minutes, hours, day of month, month, day of week; `?` = no specific value).
5. **Trigger status:** Paused — the trigger is saved but starts no runs. Switching it to Active later turns the schedule on.
6. Click **Save**.

![Schedules & Triggers dialog — scheduled trigger](../../../assets/images/a1849dbeb43b4932ac181a3d479eed65.png "Schedules & Triggers dialog — scheduled trigger")

| Setting | Value |
|---------|-------|
| Trigger type | Scheduled |
| Schedule type | Advanced — Every Day at 06:00 |
| Cron syntax (optional) | `0 0 6 * * ?` |
| Time zone | `Europe/Warsaw` |
| Trigger status | **Paused** (leave paused for the lab) |

### 5.8 Run the pipeline

Click **Run now** to start the job manually.

![Run now on the job page](../../../assets/images/final_06_run_now.png "Run now on the job page")

The tasks run in sequence — `bronze_load`, then `silver_transform`, then `gold_aggregation` — and the run ends as **Succeeded** with all three tasks green.
Click a task → **Output** to see the JSON returned by `dbutils.notebook.exit()`.

> **Tip:** If a task fails (for example an `assert` in its check cell), fix the notebook and click **Repair run**: it reruns only the failed task and the tasks after it; successful upstream tasks are not rerun.

![Run succeeded: bronze_load, silver_transform and gold_aggregation green](../../../assets/images/final_07_run_succeeded.png "Run succeeded: bronze_load, silver_transform and gold_aggregation green")

![gold_aggregation output: JSON from dbutils.notebook.exit()](../../../assets/images/final_08_task_output.png "gold_aggregation output: JSON from dbutils.notebook.exit()")

---

## 6. Verify the Gold table in SQL Editor

![SQL Editor: newest rows of gold.lab_daily_orders](../../../assets/images/final_10_sql_preview.png "SQL Editor: newest rows of gold.lab_daily_orders")

After the job run completes, open **SQL Editor** (left sidebar → SQL Editor) and run the queries below.
Replace `<catalog>` with the `catalog` value printed in section 2. Queries 1 and 2 are required; 3 and 4 are optional.

### 6.1 Required checks

```sql
-- 1. Check the table exists and preview rows
SELECT *
FROM <catalog>.gold.lab_daily_orders
ORDER BY order_date DESC
LIMIT 10;
```

```sql
-- 2. Row count — expected 32 days (2024-12-31 … 2025-01-31); rows without a valid date were removed in Silver
SELECT COUNT(*) AS total_days
FROM <catalog>.gold.lab_daily_orders;
```

![SQL Editor: total_days = 32](../../../assets/images/final_09_sql_row_count.png "SQL Editor: total_days = 32")

### 6.2 Optional checks

```sql
-- 3. (optional) Revenue sanity check — total revenue across all days
SELECT
    SUM(total_revenue)  AS grand_revenue,
    SUM(order_count)    AS grand_orders,
    MIN(order_date)     AS first_date,
    MAX(order_date)     AS last_date
FROM <catalog>.gold.lab_daily_orders;
```

```sql
-- 4. (optional) Inspect Delta version history — confirm the last write came from the pipeline job
DESCRIBE HISTORY <catalog>.gold.lab_daily_orders;
```

> **Tip:** You can also run `DESCRIBE TABLE EXTENDED <catalog>.gold.lab_daily_orders` to see the table location, schema, and Unity Catalog metadata.

---

## 7. Final checklist

### Task notebooks

- [ ] `lab_task_01_bronze` runs without error — 100,000 rows
- [ ] `lab_task_02_silver` runs without error, CHECK cells pass — 90,904 rows
- [ ] `lab_task_03_gold` runs without error, CHECK cells pass — 32 rows
- [ ] Smoke test (section 4): row_count equals expected for all 3 layers

### Job and SQL check

- [ ] Job created: `bronze_load → silver_transform → gold_aggregation`, compute Serverless (or the compute your trainer named)
- [ ] Job parameters `catalog` and `source_path` copied from the section 2 output
- [ ] Scheduled trigger: Every Day at 06:00, Europe/Warsaw (`0 0 6 * * ?`), status Paused
- [ ] Manual run succeeded — all 3 tasks green
- [ ] Task output JSON inspected in the Output tab
- [ ] Gold table verified in SQL Editor (section 6, queries 1 and 2 return data)

---

## Summary

- The Medallion pipeline keeps raw data in Bronze, typed and cleaned data in Silver, and business aggregates in Gold.
- Each layer is a notebook that reads its settings from widgets, so the job passes `catalog` and `source_path` as job parameters.
- Task dependencies run the notebooks in order; a failed `assert` fails its task and stops the tasks after it, and **Repair run** reruns only what failed.
- A scheduled trigger runs the job every day; it stays Paused until you switch it to Active.

> ← [M06: Unity Catalog](../../modules/06_unity_catalog.ipynb) | [M07: AI/BI Dashboards (optional)](../../modules/07_aibi_dashboards.ipynb) →